# Import

In [1]:
import sys

sys.path.append(r"D:\Materials\AI\Projects\Lib\torchwires\src")

from torchwires import Repo
from torchwires import BaseCallback
from torchwires import Trainer

In [2]:
import torch
from torch.utils.data import DataLoader, Dataset

import numpy as np

# Create

In [3]:
repo = Repo(
    repo_name="full_test_2",
)

# Models

In [4]:
class LinearStack(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = torch.nn.Linear(10, 10)
        self.activation = torch.nn.ReLU()

    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        return x

In [5]:
m1 = LinearStack()

repo.register_model(
    name='m1',
    model=m1
)

Model m1: has been registered
Model m1 weights not found: path=full_test_2\exp_1\last\m1.weights.pth


# Optimizers

In [6]:
repo.register_optimizer(
    name='m1',
    optimizer= torch.optim.Adam(m1.parameters(), lr=0.001)
)

Optimizer m1: has been registered
Optimizer m1 cache not found: path=full_test_2\exp_1\last\m1.optimizer.pth


# Loaders

In [7]:
class MyDataset(Dataset):
    def __init__(self):
        self.err = 0.0

    def __len__(self):
        return 30

    def __getitem__(self, index):
        x = torch.rand((10,))
        y = torch.sin(x) + self.err * torch.randn(10)
        return x, y

In [8]:
train_loader = DataLoader(
        MyDataset(),
        batch_size=2,
)

val_loader = DataLoader(
    MyDataset(),
    batch_size=2,
)

test_loader = DataLoader(
    MyDataset(),
    batch_size=2,
)

# Connections

In [9]:
repo.add_forward(
    model_name='m1',
    inputs=['x'],
    outputs=['y-hat'],
)

ForwardStep added: y-hat = m1(x)


# Losses & Metrics

In [10]:
repo.add_loss(
    loss_name='mae',
    loss_function=lambda s: torch.flatten(s['y-hat'] - s['y']).abs().mean(),
    weight_function= lambda s: 1.0
)

LossStep added: mae


In [11]:
repo.add_metric(
    metric_name='err_inverse',
    metric_function=lambda s: 1 / torch.flatten(s['y-hat'] - s['y']).abs().mean(),
)

MetricStep added: err_inverse


# Callbacks

In [12]:
trainer = Trainer(
    repo=repo,
    device='cpu'
)

In [13]:
class CB_1(BaseCallback):
    def on_epoch_start(
            self
    ):
        print("Epoch started.")

    def on_epoch_end(
            self,
            epoch_state,
    ):
        print(f"Epoch {epoch_state.aggregate_over_batches('epoch', 'train', 'mean')} ended.")
        

trainer.register_callback(CB_1())

In [14]:
trainer.register_checkpoint_callback(
    mode="min",
    monitor="total_loss",
    split="val",
)

In [15]:
trainer.register_checkpoint_callback(
    mode="max",
    monitor="err_inverse",
    split="val",
)

In [16]:
trainer.register_auto_save_callback(
    interval=5,
    checkpoint_name="auto_a",
    concat_with_epoch_no=True
)

In [17]:
trainer.register_auto_save_callback(
    interval=5,
    checkpoint_name="auto_b",
    concat_with_epoch_no=False
)

# Load

In [18]:
repo.load()

Model m1 weights not found: path=full_test_2\exp_1\last\m1.weights.pth
Optimizer m1 cache not found: path=full_test_2\exp_1\last\m1.optimizer.pth
No History cache found: path=full_test_2\exp_1\history.json
No History Features found: path=full_test_2\exp_1\history_tracked.json


# Training

In [19]:
trainer.train(
    n_epochs=20,
    train_loader=train_loader,
    val_loader=val_loader,
    loader_output_keys=['x','y']
)

Training full_test_2: Starting training epoch 0 -> 20 epochs
Epoch started.
Epoch: 1/20                                                                                                                                                                                                                                                                                               
train-mae.raw: 0.39260               | val-mae.raw: 0.39884                 | train-mae.weight: 1.00000            | val-mae.weight: 1.00000             
train-mae.eff: 0.39260               | val-mae.eff: 0.39884                 | train-err_inverse: 2.57580           | val-err_inverse: 2.55725            
train-total_loss: 0.39260            | val-total_loss: 0.39884              | train-m1.lr: 0.00100                 | val-m1.lr: None                     
Epoch 1.0 ended.
Checkpoint taken: val-total_loss improved from inf to 0.398839 | checkpoint name: best_checkpoint_val-total_loss
Model m1 weights saved: path=full

# Access Models

In [20]:
repo.get_all_models_names()

['m1']

In [21]:
repo.get_model_node('m1').get_model()

LinearStack(
  (linear1): Linear(in_features=10, out_features=10, bias=True)
  (activation): ReLU()
)

# Save

In [22]:
repo.save()

Model m1 weights saved: path=full_test_2\exp_1\last\m1.weights.pth
Optimizer m1 weights saved: path=full_test_2\exp_1\last\m1.optimizer.pth


In [23]:
repo.save(
    "final_dat"
)

Model m1 weights saved: path=full_test_2\exp_1\final_dat\m1.weights.pth
Optimizer m1 weights saved: path=full_test_2\exp_1\final_dat\m1.optimizer.pth


# Predict

In [24]:
trainer.predict(
    dataloader=test_loader,
    loader_output_keys=['x','y'],
)

Epoch: 1/1                                                                                                                                                                                                                                                                          
test-mae.raw: 0.28344                | test-mae.weight: 1.00000             | test-mae.eff: 0.28344                | test-err_inverse: 3.59407           
test-total_loss: 0.28344             | test-m1.lr: None                     | 

[{'epoch': 1,
  'batch': 1,
  'loader': 'test',
  'total_loss': tensor(0.3117),
  'x': tensor([[0.8354, 0.8253, 0.6947, 0.4381, 0.7980, 0.7754, 0.0855, 0.3064, 0.0736,
           0.6853],
          [0.4260, 0.4422, 0.0732, 0.0133, 0.9869, 0.7387, 0.3317, 0.3800, 0.8683,
           0.2337]]),
  'y': tensor([[0.7416, 0.7347, 0.6402, 0.4242, 0.7159, 0.7000, 0.0854, 0.3017, 0.0736,
           0.6329],
          [0.4133, 0.4279, 0.0731, 0.0133, 0.8343, 0.6733, 0.3257, 0.3709, 0.7632,
           0.2316]]),
  'y-hat': tensor([0.5126, 0.6303, 0.5834, 0.4403, 0.0000, 0.0000, 0.2424, 0.5033, 0.1125,
          0.5268]),
  'mae.raw': tensor(0.3117),
  'mae.weight': 1.0,
  'mae.eff': tensor(0.3117),
  'err_inverse': tensor(3.2083)},
 {'epoch': 1,
  'batch': 2,
  'loader': 'test',
  'total_loss': tensor(0.2830),
  'x': tensor([[0.4144, 0.0999, 0.4387, 0.4038, 0.0542, 0.6250, 0.9526, 0.6300, 0.2401,
           0.0835],
          [0.0482, 0.4164, 0.8097, 0.9474, 0.6821, 0.0494, 0.7947, 0.3557, 0.6860,